# 🔮 Word Predictor — Train GPT-2 on WikiText-103 (10% subset)

Fine-tunes GPT-2 small (124M) on a **10% subset of WikiText-103** (~10M tokens).
This is a good middle ground between WikiText-2 (~2M tokens) and full WikiText-103 (~100M tokens).

### Key features
- **Google Drive checkpointing** — progress is saved to Drive automatically
- **Auto-resume** — if training is interrupted, re-run the cells and it picks up where it left off
- **No data loss on disconnect** — checkpoints are written every 500 steps

**Runtime**: `Runtime → Change runtime type → T4 GPU`

## 1. Setup & Mount Drive

In [ ]:
!pip install -q torch transformers datasets tqdm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/Wordprediktor'
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints')
DRIVE_MODEL_DIR = os.path.join(DRIVE_BASE, 'models', 'gpt2-wikitext103-10pct')
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f'Drive checkpoint dir: {DRIVE_CHECKPOINT_DIR}')
print(f'Drive model dir:      {DRIVE_MODEL_DIR}')

## 2. Configuration

In [ ]:
import os, math
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)

# ---- Model ----
MODEL_NAME = 'gpt2'

# ---- Data ----
# Using 10% of WikiText-103: ~10M tokens (between wiki2's 2M and wiki103's 100M)
DATASET_NAME = 'wikitext'
DATASET_CONFIG = 'wikitext-103-raw-v1'
TRAIN_SPLIT = 'train[:10%]'       # ~10M tokens — adjust percentage as needed
VAL_SPLIT   = 'validation'         # full validation set (~250K tokens)
TEST_SPLIT  = 'test'
BLOCK_SIZE = 256

# ---- Training ----
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 5e-5
WARMUP_STEPS = 300

# ---- Checkpointing (critical for surviving interruptions) ----
SAVE_STEPS = 500              # save a checkpoint every 500 steps
EVAL_STEPS = 500              # evaluate every 500 steps
LOGGING_STEPS = 100
SAVE_TOTAL_LIMIT = 3          # keep last 3 checkpoints to save space

# Use Drive directly as output_dir so checkpoints persist across sessions
OUTPUT_DIR = DRIVE_CHECKPOINT_DIR

print('Configuration set ✓')
print(f'  Dataset: {DATASET_CONFIG} ({TRAIN_SPLIT})')
print(f'  Checkpoints saved to: {OUTPUT_DIR}')

## 3. Load & Tokenize Data

In [ ]:
print('Loading WikiText-103 (10% subset) ...')
train_raw = load_dataset(DATASET_NAME, DATASET_CONFIG, split=TRAIN_SPLIT)
val_raw   = load_dataset(DATASET_NAME, DATASET_CONFIG, split=VAL_SPLIT)
print(f'Train rows: {len(train_raw):,}   Val rows: {len(val_raw):,}')

# Combine into a DatasetDict-like structure for convenience
from datasets import DatasetDict
raw = DatasetDict({'train': train_raw, 'validation': val_raw})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=False, return_attention_mask=False)

print('Tokenizing ...')
tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['text'], desc='Tokenizing')

def group_texts(examples):
    concat = {k: sum(examples[k], []) for k in examples}
    n = (len(concat['input_ids']) // BLOCK_SIZE) * BLOCK_SIZE
    result = {k: [v[i:i+BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)] for k, v in concat.items()}
    result['labels'] = result['input_ids'].copy()
    return result

print(f'Grouping into blocks of {BLOCK_SIZE} tokens ...')
lm_dataset = tokenized.map(group_texts, batched=True, desc='Grouping')
print(f'Train blocks: {len(lm_dataset["train"]):,}   Val blocks: {len(lm_dataset["validation"]):,}')

## 4. Train with Auto-Resume

Checkpoints are saved to **Google Drive** every 500 steps.
If training is interrupted, just re-run this cell — it will automatically
find the latest checkpoint and resume from there.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    dataloader_num_workers=2,
    # Save optimizer and scheduler state so training can truly resume
    save_safetensors=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset['train'],
    eval_dataset=lm_dataset['validation'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

# ---- Auto-resume from latest checkpoint ----
resume_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith('checkpoint-')
    ]
    if ckpts:
        resume_ckpt = max(ckpts, key=os.path.getmtime)
        print(f'🔄 Resuming from checkpoint: {resume_ckpt}')
    else:
        print('🆕 No checkpoint found — starting fresh')
else:
    print('🆕 No checkpoint directory — starting fresh')

print('\nStarting training ...')
trainer.train(resume_from_checkpoint=resume_ckpt)

## 5. Evaluate & Save Final Model

In [ ]:
# Evaluate
eval_result = trainer.evaluate()
ppl = math.exp(eval_result['eval_loss'])
print(f'Validation perplexity: {ppl:.2f}')

# Save final model to a dedicated directory on Drive
trainer.save_model(DRIVE_MODEL_DIR)
tokenizer.save_pretrained(DRIVE_MODEL_DIR)
print(f'\nFinal model saved to: {DRIVE_MODEL_DIR} ✓')

## 6. Test Word Prediction

In [ ]:
import torch
from typing import List, Tuple, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM


class TransformerPredictor:
    """
    Word-completion predictor using GPT-2.
    Uses beam search over BPE tokens; a beam is 'complete' when the
    decoded new text contains a whitespace (word boundary).
    """

    def __init__(self, model_path='gpt2', device=None, max_steps=10):
        self.device = device or (
            'cuda' if torch.cuda.is_available()
            else 'mps' if torch.backends.mps.is_available()
            else 'cpu'
        )
        print(f'[Predictor] Loading {model_path} on {self.device}')
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        self.max_steps = max_steps
        self.eos_id = self.tokenizer.eos_token_id

    def predict(self, context, prefix='', top_k=5, beam_width=20):
        if not context and not prefix:
            return []
        full = context
        if prefix:
            if full and not full.endswith(' '):
                full += ' '
            full += prefix
        input_ids = self.tokenizer.encode(full, return_tensors='pt').to(self.device)
        completions = self._beam_search(input_ids, prefix, beam_width)
        seen = set()
        unique = []
        for w, s in completions:
            k = w.lower()
            if k not in seen and k:
                seen.add(k)
                unique.append((w, s))
        unique.sort(key=lambda x: x[1], reverse=True)
        return [w for w, _ in unique[:top_k]]

    def _beam_search(self, input_ids, prefix, beam_width):
        active = [(0.0, [])]  # (log_prob, appended_token_ids)
        completed = []
        pl = prefix.lower()

        for _ in range(self.max_steps):
            if not active:
                break
            cands = []
            for lp, app in active:
                if app:
                    ext = torch.tensor([app], device=self.device)
                    fids = torch.cat([input_ids, ext], dim=-1)
                else:
                    fids = input_ids
                with torch.no_grad():
                    logits = self.model(fids).logits[0, -1, :]
                log_p = torch.log_softmax(logits, dim=-1)
                vals, idxs = torch.topk(log_p, k=min(beam_width*3, log_p.size(0)))
                for i in range(vals.size(0)):
                    tid = idxs[i].item()
                    nlp = lp + vals[i].item()
                    napp = app + [tid]
                    if tid == self.eos_id:
                        w = self._word(napp)
                        if w and w.lower().startswith(pl):
                            completed.append((w, nlp))
                        continue
                    raw = self.tokenizer.decode(napp, skip_special_tokens=True)
                    text = raw.lstrip()
                    if not text:
                        cands.append((nlp, napp))
                        continue
                    if ' ' in text or '\n' in text or '\t' in text:
                        w = self._clean(text.split()[0])
                        if w and w.lower().startswith(pl):
                            completed.append((w, nlp))
                    else:
                        p = self._clean(text)
                        if not pl or p.lower().startswith(pl):
                            cands.append((nlp, napp))
            cands.sort(key=lambda x: x[0], reverse=True)
            active = cands[:beam_width]

        for lp, app in active:
            w = self._word(app)
            if w and w.lower().startswith(pl):
                completed.append((w, lp))
        return completed

    @staticmethod
    def _clean(t):
        o = []
        for c in t:
            if c.isalpha() or c in "-'":
                o.append(c)
            else:
                break
        return ''.join(o)

    def _word(self, ids):
        if not ids:
            return ''
        t = self.tokenizer.decode(ids, skip_special_tokens=True).lstrip()
        return self._clean(t.split()[0]) if t.split() else self._clean(t)

    def predict_from_text(self, typed_text, top_k=5):
        if not typed_text:
            return []
        if typed_text.endswith(' '):
            return self.predict(typed_text, '', top_k=top_k)
        parts = typed_text.rsplit(' ', 1)
        if len(parts) == 1:
            return self.predict('', parts[0], top_k=top_k)
        return self.predict(parts[0] + ' ', parts[1], top_k=top_k)

In [ ]:
predictor = TransformerPredictor(DRIVE_MODEL_DIR)

test_cases = [
    'The quick brown ',
    'The quick br',
    'Machine learning is a ',
    'I went to the ',
    'pyt',
    'Artificial intelli',
    'The president of the ',
    'In the year ',
]

for text in test_cases:
    suggestions = predictor.predict_from_text(text, top_k=5)
    print(f"  Input: '{text}'")
    print(f"  Suggestions: {suggestions}\n")

## 7. Interactive Test

In [ ]:
while True:
    text = input("\nType text (or 'quit'): ")
    if text.lower() == 'quit':
        break
    print(f"  Suggestions: {predictor.predict_from_text(text, top_k=5)}")